# Small encoder-decoder CNN

This notebook reads the fixed split, trains the small RGB semantic-segmentation CNN, and saves the best model and training history.

In [ ]:
from pathlib import Path
import json
import math
import random
import numpy as np
from PIL import Image
import tensorflow as tf

# The installed TensorFlow/Metal combination stalled at epoch boundaries.
# CPU execution is slower but was verified to complete multiple epochs.
tf.config.set_visible_devices([], 'GPU')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
PATCH_SIZE = 256
BATCH_SIZE = 8
IGNORE_LABEL = 255
NUM_CLASSES = 3
CLASS_COLORS = np.array([[199, 199, 199], [31, 119, 180], [255, 127, 14]], dtype=np.uint8)
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'project_plan.md').exists():
    PROJECT_ROOT = PROJECT_ROOT / 'final-project'
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
FIGURE_DIR = ARTIFACT_DIR / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
split_record = json.loads((ARTIFACT_DIR / 'split.json').read_text(encoding='utf-8'))
train_images = [PROJECT_ROOT / p for p in split_record['train_images']]
val_images = [PROJECT_ROOT / p for p in split_record['validation_images']]
def mask_for(path): return path.parent.parent / 'msk' / f'{path.stem}.png'
train_masks = [mask_for(p) for p in train_images]
val_masks = [mask_for(p) for p in val_images]
print('Training sources:', len(train_images), 'Validation sources:', len(val_images))


In [ ]:
def generate_patches(image_paths, mask_paths):
    for image_path, mask_path in zip(image_paths, mask_paths):
        with Image.open(image_path) as image, Image.open(mask_path) as mask:
            rgb = np.asarray(image.convert('RGB'))
            mask_rgb = np.asarray(mask.convert('RGB'))
        labels = np.full(mask_rgb.shape[:2], IGNORE_LABEL, dtype=np.uint8)
        for class_id, color in enumerate(CLASS_COLORS):
            labels[np.all(mask_rgb == color, axis=-1)] = class_id
        for y in range(0, labels.shape[0], PATCH_SIZE):
            for x in range(0, labels.shape[1], PATCH_SIZE):
                height, width = min(PATCH_SIZE, labels.shape[0] - y), min(PATCH_SIZE, labels.shape[1] - x)
                image_patch = np.zeros((PATCH_SIZE, PATCH_SIZE, 3), dtype=np.float32)
                label_patch = np.zeros((PATCH_SIZE, PATCH_SIZE), dtype=np.int32)
                weight_patch = np.zeros((PATCH_SIZE, PATCH_SIZE), dtype=np.float32)
                image_patch[:height, :width] = rgb[y:y + height, x:x + width] / 255.0
                label_patch[:height, :width] = labels[y:y + height, x:x + width]
                weight_patch[:height, :width] = 1.0
                yield image_patch, label_patch, weight_patch

def make_dataset(image_paths, mask_paths, training=False):
    dataset = tf.data.Dataset.from_generator(lambda: generate_patches(image_paths, mask_paths), output_signature=(tf.TensorSpec((PATCH_SIZE, PATCH_SIZE, 3), tf.float32), tf.TensorSpec((PATCH_SIZE, PATCH_SIZE), tf.int32), tf.TensorSpec((PATCH_SIZE, PATCH_SIZE), tf.float32)))
    if training: dataset = dataset.shuffle(64, seed=SEED, reshuffle_each_iteration=True)
    return dataset.batch(BATCH_SIZE)

def count_patches(paths):
    total = 0
    for path in paths:
        with Image.open(path) as image: total += math.ceil(image.height / PATCH_SIZE) * math.ceil(image.width / PATCH_SIZE)
    return total

train_steps = math.ceil(count_patches(train_images) / BATCH_SIZE)
val_steps = math.ceil(count_patches(val_images) / BATCH_SIZE)
train_dataset = make_dataset(train_images, train_masks, training=True).apply(tf.data.experimental.assert_cardinality(train_steps))
val_dataset = make_dataset(val_images, val_masks).apply(tf.data.experimental.assert_cardinality(val_steps))
print('Training steps:', train_steps, 'Validation steps:', val_steps)


In [ ]:
def convolution_block(inputs, filters, name):
    x = tf.keras.layers.Conv2D(filters, 3, padding='same', activation='relu', name=f'{name}_conv1')(inputs)
    return tf.keras.layers.Conv2D(filters, 3, padding='same', activation='relu', name=f'{name}_conv2')(x)

def build_small_encoder_decoder():
    inputs = tf.keras.Input((PATCH_SIZE, PATCH_SIZE, 3), name='rgb_input')
    encoder_1 = convolution_block(inputs, 16, 'encoder_1')
    encoder_2 = convolution_block(tf.keras.layers.MaxPooling2D(2)(encoder_1), 32, 'encoder_2')
    bottleneck = convolution_block(tf.keras.layers.MaxPooling2D(2)(encoder_2), 64, 'bottleneck')
    decoder_1 = convolution_block(tf.keras.layers.Concatenate()([tf.keras.layers.UpSampling2D(2)(bottleneck), encoder_2]), 32, 'decoder_1')
    decoder_2 = convolution_block(tf.keras.layers.Concatenate()([tf.keras.layers.UpSampling2D(2)(decoder_1), encoder_1]), 16, 'decoder_2')
    outputs = tf.keras.layers.Conv2D(NUM_CLASSES, 1, padding='same', activation='softmax')(decoder_2)
    return tf.keras.Model(inputs, outputs, name='small_encoder_decoder')

model = build_small_encoder_decoder()
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=tf.keras.losses.SparseCategoricalCrossentropy())
model.summary()


In [ ]:
MAX_EPOCHS = 30
CHECKPOINT_PATH = ARTIFACT_DIR / 'small_encoder_decoder_best.keras'
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, mode='min', restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(CHECKPOINT_PATH, monitor='val_loss', mode='min', save_best_only=True, verbose=1),
]
history = model.fit(train_dataset, validation_data=val_dataset, epochs=MAX_EPOCHS, callbacks=callbacks, shuffle=False)
history_values = {key: [float(value) for value in values] for key, values in history.history.items()}
(ARTIFACT_DIR / 'small_encoder_decoder_history.json').write_text(json.dumps(history_values, indent=2) + '\n', encoding='utf-8')
figure, axis = __import__('matplotlib.pyplot').pyplot.subplots(figsize=(7, 4))
axis.plot(history_values['loss'], label='Training loss')
axis.plot(history_values['val_loss'], label='Validation loss')
axis.set(xlabel='Epoch', ylabel='Sparse categorical cross-entropy', title='Small encoder-decoder training curves')
axis.legend(); axis.grid(alpha=0.3); figure.tight_layout()
figure.savefig(FIGURE_DIR / 'small_encoder_decoder_curves.png', dpi=150)
print('Saved:', CHECKPOINT_PATH)
